### Tools

Model can request to call the tools which perform the function of fetching the data from the database , webpages or running code . Tools are a pairing of:

1) a Schema including the name of the tool , a description and a argument definitions(a JSON schema).
2) a function or coroutine to execute

In [4]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

# Load variables from your .env file
load_dotenv()

# explicitly pull the key using your custom name from the .env file
gemini_api_key = os.getenv("Gemini_API_Key")

# Initialize the Gemini model, passing the key explicitly
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=gemini_api_key
)

@tool 
def get_weather(location: str) -> str:
    """ Get the weather of the location """
    return f"The weather of {location} is sunny" 

# Bind the tool to the model
model_with_tools = model.bind_tools([get_weather])


In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'a5d47981-7381-4cfa-97aa-e2f2ea1bf0bc': 'CukBAQw51sepqck/hj3eyIeM2Y5c35DrQ9TFHV0XDWS5Yk2uf22VCL9Mwrs6naqiuo+v/NL+HGG3qoqU7otmJh5qILyKX5rvY4ZbxHUuoXORC25X8LheJ5k0b9xu2UQqSmFrrdSGWgtTMgwBav+Zmd8ZLLfHXNDHaVwDykBs3pQSq+G5UXE1FNlEmM11nY3UvhzLnyZj2X26zhm24GCvk9SnkFcGhWpZP9bsTYOHVq49TDk64RGKkRvCurlzJAxBt++Cp3xoh7mutabSTe39K5jTEBUkvJmUnQuMcBnd0+KJk9pkvYjWkMf9gtk='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f09ac-2f08-7c70-962b-7bac5f7e68bb-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'a5d47981-7381-4cfa-97aa-e2f2ea1bf0bc', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 61, 'total_tokens': 109, 'input_token_details': {'cache_read': 0}, 'output_token_de

### Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'711080f5-bed6-4d61-803f-a9a3f9bf13fc': 'CukBAQw51scyPqfsgFQs+IsefTh0E/SbALHSJBejvIXttKHTZ2V4T8HV0mljdz5A+cyV3FrbTl6Os5a5x10x/VMsdalqmpaoil5RL/yXTBxlke0ClbfW2Is5REtTxwuR+JHXMGIVnOxPQ4dgJZuTNMUWBZdiqxODC6GFaCvwqfjpbWWr75iPKPkUVuC3sQnmMA9LiG3oSCXpy1WWoycmT/V5p9baG2w5cJ99yfLePEO/LTKVhwKhbEaFVIXclExV69kv/BAn4IQsspYiQKtZlvyZVlKnMyHrC7deeXomPuYAKzms7kob4kG01MU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f09af-5816-78e3-9a19-1d196be27f32-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '711080f5-bed6-4d61-803f-a9a3f9bf13fc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 61,